# 05 - Mitochondrial DNA lineages in the KZ-219 cohort

![cohort](https://img.shields.io/badge/cohort-KZ--219-1b9e77)
![platform](https://img.shields.io/badge/platform-DNBSEQ--G400-7570b3)
![pipeline](https://img.shields.io/badge/DRAGEN-v3.10.16-d95f02)
![reference](https://img.shields.io/badge/reference-GRCh37%2Fhg19-e7298a)
![annotation](https://img.shields.io/badge/ANNOVAR-v2025Mar02-666666)
![bioproject](https://img.shields.io/badge/BioProject-PRJNA1224597-333333)

| | |
|---|---|
| **Cohort** | KZ-219 - 219 healthy Kazakh adults |
| **Sequencing** | DNBSEQ-G400, PCR-free, 100 bp paired-end |
| **Alignment and variant calling** | Illumina DRAGEN Bio-IT Platform v3.10.16 |
| **Reference** | GRCh37 / hg19 (DRAGEN hash table v8) |
| **Annotation** | ANNOVAR v2025Mar02 |
| **mtDNA calling** | reads remapped to rCRS (NC_012920.1), mutserve + haplogrep3 (`phylotree-fu-rcrs@1.2`) |
| **Raw data** | NCBI SRA, BioProject PRJNA1224597 |

### Manuscript items produced here

| Item | File |
|---|---|
| Fig. 6 | `Fig6_chrM_composition.tiff` / `.png` |
| Supplementary Fig. 8 | `Supplementary_Fig_8_chrM_call_quality.tiff` / `.png` |
| Supplementary Tables 24-26 | `Supplementary_Tables_05.docx` |
| Supplementary Data 5 | `Supplementary_Data_5_chrM.xlsx` |

### Running

| Variable | Default | Content |
|---|---|---|
| `KZ_CHRM_DIR` | `data/chrM` | `kz_219_mtdna_haplogroups.tsv` (haplogrep3 `--extend-report`) and `kz_219_chrM_metadata_final.xlsx` (maternal tribe and zhuz) |
| `KZ_OUT_DIR` | `results/05_chrM` | figures, Supplementary Data 5, Supplementary Tables, and `kz_219_chrM_lineage_long.tsv` (long-format calls for the paired mtDNA and chrY comparison) |

```bash
export KZ_CHRM_DIR=data/chrM
pip install -r requirements.txt
jupyter nbconvert --to notebook --execute --inplace 05_chrM_analysis.ipynb
```

Upstream, mitochondrial reads were extracted from each GRCh37 alignment, realigned to the rCRS
with BWA-MEM, called with mutserve and classified with haplogrep3 against
`phylotree-fu-rcrs@1.2` (Methods). The metadata file holds self-reported tribal affiliation and
is not part of the repository.

Figures are written as TIFF (CMYK, LZW) and PNG (RGB) at 1200 dpi, Arial 8 pt with 10 pt
panel letters.

## 1 - Setup

In [ ]:
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.stats import binomtest, fisher_exact

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

import scipy
import openpyxl
print('pandas', pd.__version__, '| numpy', np.__version__,
      '| scipy', scipy.__version__, '| matplotlib', plt.matplotlib.__version__,
      '| openpyxl', openpyxl.__version__)

In [ ]:
MM = 1 / 25.4
W1, W15, W2 = 88 * MM, 120 * MM, 180 * MM

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'Liberation Sans', 'DejaVu Sans'],
    'font.size': 8, 'axes.labelsize': 8, 'axes.titlesize': 8,
    'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'legend.title_fontsize': 8, 'hatch.linewidth': 0.5,
    'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6,
    'xtick.major.size': 2, 'ytick.major.size': 2,
    'axes.edgecolor': '#3C3C3C', 'xtick.color': '#3C3C3C', 'ytick.color': '#3C3C3C',
    'text.color': '#1A1A1A', 'axes.labelcolor': '#1A1A1A',
    'grid.linewidth': 0.4, 'grid.color': '#C8C8C8',
    'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none',
    'figure.facecolor': 'white', 'savefig.facecolor': 'white',
    'savefig.bbox': 'tight', 'savefig.pad_inches': 0.02,
})

INK, RULE, FAINT, BAND = '#1A1A1A', '#3C3C3C', '#B0B0B0', '#4A4A4A'
FS, FS_PANEL, FS_MAX = 8, 10, 12
FIG_FORMATS = ('tiff', 'png')          
ANC_HATCH  = {'East': None, 'West': '////', 'Unassigned': '....'}
ANC_MARKER = {'East': 'o', 'West': 's', 'Unassigned': '^'}

MAIN_FIG = {'chrM_composition': 6}
SUPP_NUM = {'fig_call_quality': 8, 'tab_haplogroups': 24, 'tab_composition': 25,
            'tab_diversity': 26, 'data': 5}

ANC_ORDER  = ['East', 'West', 'Unassigned']
ANC_COLORS = {'East': '#2C6E9B', 'West': '#B5502F', 'Unassigned': '#A0A0A0'}
ANC_LABEL  = {'East': 'East Eurasian', 'West': 'West Eurasian',
              'Unassigned': 'Undetermined origin'}
HEAT = mcolors.LinearSegmentedColormap.from_list(
    'kz_slate', ['#FFFFFF', '#DCE4EA', '#AEBECB', '#6E8598', '#3A4E60', '#1B2A36'],
    N=256)


def on(hex_color):
    r, g, b = mcolors.to_rgb(hex_color)
    return '#FFFFFF' if 0.299 * r + 0.587 * g + 0.114 * b < 0.55 else INK


def clade_palette(clades, anc_of):
    ramps = {'East': ('#12405F', '#C3D8E5'), 'West': ('#7A3418', '#EDCBB8'),
             'Unassigned': ('#8A8A8A', '#D8D8D8')}
    out = {}
    for a in ANC_ORDER:
        sub = [c for c in clades if anc_of.get(c) == a]
        if not sub:
            continue
        cm = mcolors.LinearSegmentedColormap.from_list('ramp', list(ramps[a]))
        for i, c in enumerate(sub):
            out[c] = mcolors.to_hex(cm(i / max(1, len(sub) - 1)))
    return out


def panel(ax, letter, dx=-0.06, dy=1.02):
    ax.text(dx, dy, letter, transform=ax.transAxes, fontsize=FS_PANEL, fontweight='bold',
            va='bottom', ha='left', color=INK)


def despine(ax, keep=('left', 'bottom')):
    for name, sp in ax.spines.items():
        sp.set_visible(name in keep)


def _enforce_text(fig):
    """Raise any text below 8 pt (cap above 12 pt) and report what changed."""
    changed = []
    for t in fig.findobj(plt.matplotlib.text.Text):
        if not t.get_visible() or not t.get_text().strip():
            continue
        s = t.get_fontsize()
        if s < FS - 1e-6 or s > FS_MAX + 1e-6:
            changed.append((t.get_text()[:24], round(s, 2)))
            t.set_fontsize(min(max(s, FS), FS_MAX))
    if changed:
        print(f'  NOTE: {len(changed)} text item(s) raised to {FS} pt, check the layout: '
              f'{changed[:6]}')
    return changed


def save_fig(fig, stem, outdir='.', dpi=1200, formats=FIG_FORMATS):
    """PNG (RGB) and TIFF (CMYK, LZW) at 1200 dpi, named with the figure number."""
    from PIL import Image
    _enforce_text(fig)
    png = os.path.join(outdir, f'{stem}.png')
    fig.savefig(png, dpi=dpi, bbox_inches='tight', pad_inches=0.03)
    paths = []
    if 'tiff' in formats:
        tif = os.path.join(outdir, f'{stem}.tiff')
        with Image.open(png) as im:
            im.convert('RGB').convert('CMYK').save(tif, compression='tiff_lzw', dpi=(dpi, dpi))
        paths.append(tif)
    if 'png' in formats:
        paths.append(png)
    else:
        os.remove(png)
    for p in paths:
        with Image.open(p) as im:
            w, h, mode = *im.size, im.mode
        wmm = w / dpi * 25.4
        print(f'saved {os.path.basename(p)}: {mode}, {wmm:.0f} x {h / dpi * 25.4:.0f} mm '
              f'at {dpi} dpi, {os.path.getsize(p) / 1e6:.1f} MB'
              + ('   WARNING: wider than 183 mm' if wmm > 183 else ''))
    return paths


HEAD_FILL = PatternFill(fill_type='solid', start_color='FFDDEBF7', end_color='FFDDEBF7')
HEAD_FONT = Font(name='Arial', size=10, bold=True, color='FF1F3864')
BODY_FONT = Font(name='Arial', size=10)
RULE_B = Border(bottom=Side(style='thin', color='FFBFBFBF'))


def write_sheet(ws, frame, key_cols=()):
    """Write a frame with a formatted header; key columns left-aligned, the rest centred."""
    cols = list(frame.columns)
    ws.append([str(c) for c in cols])
    for row in frame.itertuples(index=False):
        ws.append([None if (isinstance(v, float) and np.isnan(v)) else
                   (v.item() if hasattr(v, 'item') else v) for v in row])
    for j, c in enumerate(cols, start=1):
        h = ws.cell(row=1, column=j)
        h.font, h.fill, h.border = HEAD_FONT, HEAD_FILL, RULE_B
        h.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        left = c in key_cols
        for r in range(2, ws.max_row + 1):
            cell = ws.cell(row=r, column=j)
            cell.font = BODY_FONT
            cell.alignment = Alignment(horizontal='left' if left else 'center',
                                       vertical='center')
        width = max([len(str(c))] + [len(str(ws.cell(row=r, column=j).value or ''))
                                     for r in range(2, min(ws.max_row, 500) + 1)])
        ws.column_dimensions[get_column_letter(j)].width = min(max(width + 3, 10), 44)
    ws.row_dimensions[1].height = 30
    ws.freeze_panes = 'A2'


def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan, np.nan)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z / d * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return p, max(0.0, c - h), min(1.0, c + h)

## 2 - Configuration

In [ ]:
WORK_DIR = os.environ.get('KZ_CHRM_DIR', 'data/chrM')       # haplogrep3 report and metadata
OUT_DIR  = os.environ.get('KZ_OUT_DIR', 'results/05_chrM')

HG_FILE   = 'kz_219_mtdna_haplogroups.tsv'          # haplogrep3 --extend-report
META_FILE = 'kz_219_chrM_metadata_final.xlsx'       # maternal tribe / zhuz
Y_FILE    = 'kz_219_chrY_lineage_long.tsv' 

QUALITY_THRESHOLD = 0.90    # haplogrep3 score below which the terminal label is flagged
MIN_TRIBE_N       = 10      # tribes below this are pooled; no per-tribe rate is reported
DPI               = 1200

XLSX_OUT = f"Supplementary_Data_{SUPP_NUM['data']}_chrM.xlsx"
LONG_OUT = 'kz_219_chrM_lineage_long.tsv'

assert os.path.isdir(WORK_DIR), f'input folder not found: {WORK_DIR} (set KZ_CHRM_DIR)'
os.makedirs(OUT_DIR, exist_ok=True)
print('WORK_DIR:', WORK_DIR)
print('OUT_DIR :', OUT_DIR)

In [ ]:
def find_file(name, extra_dirs=()):
    if os.path.isabs(name) and os.path.exists(name):
        return name
    for d in [WORK_DIR, '.', OUT_DIR, 'data', *extra_dirs]:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'{name!r} not found; check the Configuration cell.')


def read_table(path, **kw):
    path = find_file(path)
    with open(path, encoding='utf-8-sig', errors='replace') as fh:
        head = fh.readline()
    sep = max([',', ';', '\t', '|'], key=head.count)
    df = pd.read_csv(path, sep=sep, encoding='utf-8-sig', **kw)
    df.columns = [str(c).strip() for c in df.columns]
    return df


def pick_col(df, *names):
    low = {str(c).lower().strip(): c for c in df.columns}
    for n in names:
        if n in low:
            return low[n]
    return None


def rename_unique(df, mapping):
    mapping = {k: v for k, v in mapping.items() if k is not None}
    out = df.copy()
    clash = [v for k, v in mapping.items() if v in out.columns and v != k]
    out = out.drop(columns=clash).rename(columns=mapping)
    return out.loc[:, ~out.columns.duplicated()]

In [ ]:
def blocks(h):
    s = str(h).strip().rstrip('~*!')
    s = s.split('+')[0].split('(')[0].strip()
    return re.findall(r'[A-Za-z]+|\d+', s)


def collapse_hg(h, level=None):
    b = blocks(h)
    if not b:
        return str(h)
    return ''.join(b[:level]) if level else ''.join(b)

_MT_TWO_CHAR = set('CDGBNTUKZAYWXIFVHJR')

def get_major_mt(h):
    # Major clade used on the x axis of Fig. 5a: 'D4m2a' -> 'D4', 'HV2a' -> 'HV2'.
    s = collapse_hg(h)
    if s.startswith('HV'):
        return 'HV2' if s.startswith('HV2') else 'HV'
    b = blocks(h)
    if not b:
        return '?'
    if len(b) > 1 and b[0] in _MT_TWO_CHAR and b[1].isdigit():
        cand = b[0] + b[1]
        return cand if cand in MT_CLADE_ORDER else b[0]
    return b[0]


def mt_clade(h):
    # Coarser level for the tribe matrices: one column per clade, one ancestry per column.
    b = blocks(h)
    if not b:
        return '?'
    L = b[0]
    d = b[1] if len(b) > 1 and b[1].isdigit() else ''
    if L == 'HV':
        return 'HV'
    if L == 'N':
        return 'N1' if d == '1' else ('N9' if d == '9' else 'N11')
    if L == 'R':
        return 'R9' if d == '9' else ('R1' if d == '1' else 'R')
    if L == 'B':
        return 'B' + d if d in ('4', '5') else 'B'
    if L == 'Y':
        return 'Y'
    return L


# R1/R9 and N11 are kept apart from the R and N paragroups: they differ in
# phylogeographic origin, so a merged bar could not carry a single colour.
MT_CLADE_ORDER = ['M', 'C4', 'C5', 'C7', 'C', 'D4', 'D5', 'D', 'G1', 'G2', 'G',
                  'Z3', 'Z', 'N1', 'N9', 'N11', 'N', 'A8', 'A', 'Y1', 'W', 'X2', 'I',
                  'R1', 'R9', 'R', 'B4', 'B5', 'F1', 'F2', 'H1', 'H3', 'H', 'HV2',
                  'HV', 'V', 'J1', 'J2', 'T1', 'T2', 'U2', 'U3', 'U4', 'U5', 'K']
MT_MACRO_OF = {**{c: 'M' for c in ['M', 'C4', 'C5', 'C7', 'C', 'D4', 'D5', 'D',
                                   'G1', 'G2', 'G', 'Z3', 'Z']},
               **{c: 'N' for c in ['N1', 'N9', 'N11', 'N', 'A8', 'A', 'Y1', 'W',
                                   'X2', 'I']},
               **{c: 'R' for c in ['R1', 'R9', 'R', 'B4', 'B5', 'F1', 'F2', 'H1', 'H3', 'H',
                                   'HV2', 'HV', 'V', 'J1', 'J2', 'T1', 'T2',
                                   'U2', 'U3', 'U4', 'U5', 'K']}}
MT_MACRO_LABEL = {'M': 'macro-M', 'N': 'macro-N', 'R': 'macro-R'}
PARAGROUPS = {'M', 'C', 'D', 'G', 'Z', 'N', 'A', 'R', 'H', 'HV'}

MT_MATRIX_COLS = ['M', 'C', 'D', 'G', 'Z', 'N9', 'N11', 'A', 'Y', 'B4', 'B5', 'F',
                  'R9', 'N1', 'W', 'X', 'I', 'R1', 'H', 'HV', 'V', 'J', 'T', 'U',
                  'K', 'R']
Y_MATRIX_COLS = ['C2a', 'C2b', 'O2', 'N1a', 'Q1b', 'R1a', 'R1b', 'R2', 'G1', 'J2', 'L1']

# Assigned per haplogroup, not per macro-haplogroup: macro-R carries both West-Eurasian
# (H, HV, V, J, T, U, K, R0, R1, R2) and East-Eurasian (B, F/R9) lineages, and macro-N
# likewise splits into A, Y, N9, N11 (East) and N1, N2/W, X, I (West).
_MT_LETTER = {'M': 'East', 'C': 'East', 'D': 'East', 'G': 'East', 'Z': 'East',
              'E': 'East', 'Q': 'East', 'A': 'East', 'Y': 'East', 'B': 'East',
              'F': 'East', 'W': 'West', 'X': 'West', 'I': 'West', 'H': 'West',
              'V': 'West', 'J': 'West', 'T': 'West', 'U': 'West', 'K': 'West'}
_N_EAST, _N_WEST = {'9', '11', '21', '22'}, {'1', '2', '5'}
_R_EAST = {'9', '11', '22', '23', '24', '30', '31'}
_R_WEST = {'0', '1', '2', '5', '6', '7', '8'}

def mt_ancestry(h):
    b = blocks(h)
    if not b:
        return 'Unassigned'
    L = b[0]
    d = b[1] if len(b) > 1 and b[1].isdigit() else None
    if L == 'HV':
        return 'West'
    if L == 'N':
        return 'East' if d in _N_EAST else ('West' if d in _N_WEST else 'Unassigned')
    if L == 'R':
        return 'East' if d in _R_EAST else ('West' if d in _R_WEST else 'Unassigned')
    return _MT_LETTER.get(L, 'Unassigned')


_Y_LETTER = {'C': 'East', 'D': 'East', 'O': 'East', 'N': 'East', 'Q': 'East',
             'G': 'West', 'J': 'West', 'L': 'West', 'T': 'West', 'E': 'West',
             'I': 'West', 'H': 'West'}

def y_ancestry(h):
    b = blocks(h)
    if not b:
        return 'Unassigned'
    return 'West' if b[0] == 'R' else _Y_LETTER.get(b[0], 'Unassigned')


def ancestry_of(df, key):
    # Every plotted group must map to exactly one origin, otherwise its bar or column
    # would be drawn in a colour that only part of it has.
    a = df.groupby(key, observed=True)['Ancestry'].agg(set)
    mixed = a[a.map(len) > 1]
    if len(mixed):
        raise ValueError(f'{key} groups carrying more than one origin: '
                         f'{ {k: sorted(v) for k, v in mixed.items()} }')
    return a.map(lambda s: next(iter(s)))

_NULLS = {'', 'nan', 'none', 'na', 'n/a', '?', 'unknown', 'no', '-'}
_TRIBE_ALIAS = {'kirgyz': 'Qyrgyz', 'kirgiz': 'Qyrgyz', 'qyrgyz': 'Qyrgyz',
                'qozha': 'Qoja', 'kozha': 'Qoja', 'khoja': 'Qoja', 'qoja': 'Qoja',
                'kerei': 'Kerey', 'kerey': 'Kerey', 'kypchak': 'Kypshak',
                'qypshaq': 'Kypshak', 'kypshak': 'Kypshak', 'nayman': 'Naiman',
                'tore': 'Tore', 'nogai': 'Nogai', 'russian': 'Russian'}
_ZHUZ_ALIAS = {'older': 'Uly', 'senior': 'Uly', 'elder': 'Uly', 'uly': 'Uly',
               'middle': 'Orta', 'midlle': 'Orta', 'orta': 'Orta',
               'junior': 'Kishi', 'younger': 'Kishi', 'kishi': 'Kishi',
               'qoja': 'Qoja', 'kozha': 'Qoja', 'tore': 'Tore',
               'qyrgyz': 'Qyrgyz', 'nogai': 'Nogai'}


def norm_tribe(t):
    t = str(t).strip().rstrip('?').strip()
    if t.lower() in _NULLS:
        return 'NA'
    t = re.split(r'\s*(?:->|\u2192|,|/)\s*', t)[0].strip().rstrip('?').strip()
    return 'NA' if t.lower() in _NULLS else _TRIBE_ALIAS.get(t.lower(), t.capitalize())


def norm_zhuz(z, tribe='NA'):
    special = {'Tore': 'Tore', 'Qoja': 'Qoja', 'Qyrgyz': 'Qyrgyz', 'Nogai': 'Nogai'}
    if tribe in special:
        return special[tribe]
    z = str(z).strip().lower()
    if z in _NULLS:
        return 'NA'
    return 'Other' if z == 'other' else _ZHUZ_ALIAS.get(z, 'NA')

## 3 - Load and harmonise

haplogrep3 calls are merged with the cohort metadata; tribe and zhuz labels are mapped to
the shared vocabulary used by the Y-chromosome notebook.

In [ ]:
hg = read_table(HG_FILE)
mt = rename_unique(hg, {pick_col(hg, 'sampleid_clean', 'sampleid', 'sample',
                                 'input_sample'): 'SampleID',
                        pick_col(hg, 'haplogroup', 'hg'): 'Haplogroup'})
mt['SampleID'] = mt['SampleID'].astype(str).str.strip()

meta_path = find_file(META_FILE)
meta = (pd.read_excel(meta_path) if meta_path.lower().endswith(('.xlsx', '.xlsm', '.xls'))
        else read_table(META_FILE))
meta.columns = [str(c).strip() for c in meta.columns]
keep = {pick_col(meta, 'sampleid', 'sampleid_clean', 'sample_id', 'sample'): 'SampleID',
        pick_col(meta, 'mother_tribe', 'mother tribe', 'maternal_tribe'): 'tribe_raw',
        pick_col(meta, 'mother_zhuz', 'mother zhuz', 'maternal_zhuz'): 'zhuz_raw'}
m = rename_unique(meta[list(keep)], keep)
m['SampleID'] = m['SampleID'].astype(str).str.strip()
mt = mt.merge(m, on='SampleID', how='left')

mt['Tribe'] = mt['tribe_raw'].apply(norm_tribe)
mt['Zhuz'] = [norm_zhuz(z, t) for z, t in zip(mt['zhuz_raw'], mt['Tribe'])]

mt['Terminal'] = mt['Haplogroup'].apply(collapse_hg)
mt['Major'] = mt['Terminal'].apply(get_major_mt)
mt['Clade'] = mt['Terminal'].apply(mt_clade)
mt['Ancestry'] = mt['Terminal'].apply(mt_ancestry)

c_qual = pick_col(mt, 'quality', 'rank')
if c_qual:
    mt['Quality'] = pd.to_numeric(mt[c_qual], errors='coerce')
for src, dst in [(('found_polys', 'foundpolys'), 'nFound'),
                 (('not_found_polys', 'notfoundpolys'), 'nNotFound'),
                 (('remaining_polys', 'remainingpolys'), 'nRemaining')]:
    col = pick_col(mt, *src)
    if col:
        mt[dst] = mt[col].map(lambda v: 0 if str(v).strip() in ('nan', '', '-')
                              else len(re.split(r'[,\s]+', str(v).strip())))

LONG_COLS = ['SampleID', 'Haplogroup', 'Terminal', 'Major', 'Clade', 'Ancestry',
             'Tribe', 'Zhuz']
mt_long = mt[LONG_COLS].copy()
mt_long['Source'] = 'chrM'
mt_long.to_csv(os.path.join(OUT_DIR, LONG_OUT), sep='\t', index=False)

N = len(mt)
print(f'mtDNA calls        : {N}')
print(f'metadata matched   : {mt["tribe_raw"].notna().sum()} / {N}')
print(f'terminal labels    : {mt["Terminal"].nunique()}')
print(f'major clades       : {mt["Major"].nunique()}')
print(f'maternal tribes    : {(mt["Tribe"] != "NA").sum()} annotated, '
      f'{(mt["Tribe"] == "NA").sum()} not recorded')

In [ ]:
tab = (mt.groupby('Ancestry', observed=True)
         .agg(n=('SampleID', 'size'), clades=('Major', 'nunique'))
         .reindex(ANC_ORDER).dropna(how='all'))
tab['pct'] = (tab['n'] / N * 100).round(1)
print('Phylogeographic origin of maternal lineages')
print(tab.to_string())

print('\nMacro-haplogroup')
macro = mt['Major'].map(MT_MACRO_OF).value_counts().reindex(['M', 'N', 'R']).dropna()
for k, v in macro.items():
    print(f'  macro-{k}: {int(v):3d} ({v / N * 100:4.1f}%)')

print('\nMaternal tribe')
tr = mt['Tribe'].value_counts()
for k, v in tr.items():
    print(f'  {k:<12} {v:3d} ({v / N * 100:4.1f}%)')
print(f'\nTribes with n >= {MIN_TRIBE_N}: '
      f'{[t for t in tr.index if t != "NA" and tr[t] >= MIN_TRIBE_N]}')

## 4 - Fig. 6a, b - haplogroup spectrum and ancestry composition

Two encodings are used and they are independent. Colour is the phylogeographic origin of
the haplogroup, assigned per clade rather than per macro-haplogroup; grey marks clades whose
macro-haplogroup contains both East- and West-Eurasian lineages and whose call is not deep
enough to choose between them. An asterisk marks a paragroup, that is, lineages of the named
clade not assigned to any subclade shown, and says nothing about origin.

Panel b reports composition only for tribes with n >= 3; the remainder are pooled, and
95% Wilson intervals are given for the East-Eurasian fraction.

West Eurasian bars and segments are also hatched, so origin does not rest on colour alone.
Clade names inside the segments of b are written only where the segment is wide enough at
8 pt; every clade is in Supplementary Data 5, sheet `03_tribe_by_clade`.

In [ ]:
MIN_TRIBE_B = 3                      
H_A, H_ROW, GAP, TOP_IN, BOT_IN = 1.80, 0.34, 0.75, 0.30, 0.42

freq = mt['Major'].value_counts()
freq = freq.reindex([c for c in MT_CLADE_ORDER if c in freq.index])
anc_of = ancestry_of(mt, 'Major')

tn = mt['Tribe'].value_counts()
BIG = [t for t in tn.index if t != 'NA' and tn[t] >= MIN_TRIBE_B]
OTHER, NOREC = f'Other tribes, each n < {MIN_TRIBE_B}', 'Tribe not recorded'
groups = ([('Whole cohort', mt)] + [(t, mt[mt['Tribe'] == t]) for t in BIG] +
          [(OTHER, mt[(~mt['Tribe'].isin(BIG)) & (mt['Tribe'] != 'NA')]),
           (NOREC, mt[mt['Tribe'] == 'NA'])])
groups = [(nm, s) for nm, s in groups if len(s)]       
print(f'  panel b rows: {len(groups)}  (tribes n >= {MIN_TRIBE_B}: {len(BIG)})')

H_B   = H_ROW * len(groups)
FIG_H = TOP_IN + H_A + GAP + H_B + BOT_IN
fig = plt.figure(figsize=(W2, FIG_H))
gs = fig.add_gridspec(2, 1, height_ratios=[H_A, H_B],
                      hspace=GAP / ((H_A + H_B) / 2),
                      left=0.165, right=0.855,
                      top=1 - TOP_IN / FIG_H, bottom=BOT_IN / FIG_H)

gsa = gs[0].subgridspec(2, 1, height_ratios=[0.08, 1.0], hspace=0.04)
axb, ax = fig.add_subplot(gsa[0]), fig.add_subplot(gsa[1])

nc = len(freq)
macro = [MT_MACRO_OF[c] for c in freq.index]
prev, start, cuts = None, 0, []
for j, mm in enumerate(macro + [None]):
    if mm != prev:
        if prev is not None:
            axb.fill_betweenx([0, 1], start - .5, j - .5, color=BAND, lw=0)
            axb.text((start + j - 1) / 2, .5, MT_MACRO_LABEL[prev], ha='center',
                     va='center', fontsize=FS, fontweight='bold', color='white')
            if start:
                cuts.append(start)
        start = j
    prev = mm
for j in cuts:
    axb.axvline(j - .5, color='white', lw=1.1)
axb.set(xlim=(-.5, nc - .5), ylim=(0, 1), xticks=[], yticks=[])
axb.spines[:].set_visible(False)

ax.bar(range(nc), freq.values, width=.76, lw=.4, edgecolor='white', zorder=3,
       color=[ANC_COLORS[anc_of[c]] for c in freq.index],
       hatch=[ANC_HATCH[anc_of[c]] for c in freq.index])
for i, v in enumerate(freq.values):
    ax.text(i, v + freq.max() * .03, str(v), ha='center', va='bottom', fontsize=FS,
            color=INK)
ax.set_xticks(range(nc))
ax.set_xticklabels([c + '*' if c in PARAGROUPS else c for c in freq.index],
                   rotation=90, fontsize=FS)
ax.set_xlim(-.5, nc - .5)
ax.set_ylim(0, freq.max() * 1.22)
ax.set_ylabel(f'Individuals (n = {N})')
ax.set_xlabel('mtDNA haplogroup', labelpad=2)
ax.grid(axis='y', zorder=0)
ax.set_axisbelow(True)
despine(ax)
for j in cuts:
    ax.axvline(j - .5, color=FAINT, lw=.5, zorder=1)
ax.legend([plt.Rectangle((0, 0), 1, 1, fc=ANC_COLORS[a], ec='white', hatch=ANC_HATCH[a])
           for a in ANC_ORDER],
          [ANC_LABEL[a] for a in ANC_ORDER], frameon=False, ncol=3, loc='upper right',
          bbox_to_anchor=(1.005, 1.045), handlelength=1.0, handleheight=.75,
          columnspacing=1.1, handletextpad=.45)
panel(axb, 'a', dx=-0.215, dy=1.6)

ax2 = fig.add_subplot(gs[1])
cols = [c for c in MT_MATRIX_COLS if c in set(mt['Clade'])]
anc_clade = ancestry_of(mt, 'Clade')
pal = clade_palette(cols, anc_clade)
SEG_MIN_PCT = 2.6     # width in % per character needed for an 8 pt label inside a segment
zof = mt[mt['Tribe'].isin(BIG)].groupby('Tribe', observed=True)['Zhuz'].agg(
    lambda s: s.mode().iat[0])

stat = [(len(s), int((s['Ancestry'] == 'East').sum()), int((s['Ancestry'] == 'West').sum()))
        for _, s in groups]
comp = np.array([[(s['Clade'] == c).sum() / len(s) * 100 for c in cols] for _, s in groups])
yy = np.arange(len(groups))[::-1]
left = np.zeros(len(groups))
for j, c in enumerate(cols):
    v = comp[:, j]
    ax2.barh(yy, v, left=left, height=.60, color=pal[c], lw=.3, edgecolor='white', zorder=3,
             hatch=ANC_HATCH[anc_clade[c]])
    for yv, vv, lv in zip(yy, v, left):
        if vv >= SEG_MIN_PCT * max(len(c), 1):
            ax2.text(lv + vv / 2, yv, c, ha='center', va='center', fontsize=FS,
                     color=on(pal[c]), zorder=4,
                     bbox=dict(boxstyle='square,pad=0.05', fc=pal[c], ec='none'))
    left += v

tr = ax2.get_yaxis_transform()
ax2.text(1.015, yy[0] + .62, 'East Eurasian,\n% (95% CI)', transform=tr, ha='left',
         va='bottom', fontsize=FS, color=BAND, linespacing=1.0)
for yv, (n, k, w) in zip(yy, stat):
    _, lo, hi = wilson(k, n)
    ax2.text(1.015, yv, f'{k / n * 100:.0f}  ({lo * 100:.0f}-{hi * 100:.0f})',
             transform=tr, ha='left', va='center', fontsize=FS, color=INK)

labels = []
for (name, _), (n, _, _) in zip(groups, stat):
    z = zof.get(name)
    labels.append(f'{name}' + (f' ({z})' if isinstance(z, str) and z != 'NA' else '') +
                  f'\nn = {n}')
ax2.set_yticks(yy)
ax2.set_yticklabels(labels, fontsize=FS, linespacing=1.0)
ax2.set_xlim(0, 100)
ax2.set_ylim(-.6, len(groups) - .4)
ax2.set_xticks(range(0, 101, 20))
ax2.set_xlabel('Maternal lineages (%)', labelpad=2)
ax2.tick_params(axis='y', length=0)
despine(ax2, keep=('bottom',))
panel(ax2, 'b', dx=-0.215, dy=1.03)

save_fig(fig, f"Fig{MAIN_FIG['chrM_composition']}_chrM_composition", OUT_DIR, dpi=DPI)
plt.show()

In [ ]:
# Supplementary Fig. 8 - haplogrep3 call quality
fig = plt.figure(figsize=(W2, 2.9))
gs = fig.add_gridspec(1, 2, width_ratios=[1.0, 1.55], wspace=.20,
                      left=.08, right=.995, top=.88, bottom=.26)

ax = fig.add_subplot(gs[0])
s = mt.sort_values('Quality').reset_index(drop=True)
for a_ in ANC_ORDER:
    m_ = (s['Ancestry'] == a_).to_numpy()
    if m_.any():
        ax.scatter(np.flatnonzero(m_), s.loc[m_, 'Quality'], s=9, lw=.25, edgecolors='white',
                   zorder=3, color=ANC_COLORS[a_], marker=ANC_MARKER[a_])
ax.axhline(QUALITY_THRESHOLD, color=RULE, ls=(0, (3, 1.6)), lw=.7, zorder=2)
ax.text(len(s) * .98, QUALITY_THRESHOLD - .012, f'quality = {QUALITY_THRESHOLD:.2f}',
        ha='right', va='top', fontsize=FS, color=RULE)
ax.set_xlim(-4, len(s) + 4)
ax.set_ylim(min(.70, s['Quality'].min() - .02), 1.015)
ax.set_ylabel('haplogrep3 quality score')
ax.set_xlabel(f'Individuals (n = {len(s)}), ranked by quality', labelpad=2)
ax.grid(axis='y', zorder=0)
ax.set_axisbelow(True)
despine(ax)
n_lo = int((mt['Quality'] < QUALITY_THRESHOLD).sum())
ax.text(.03, .97, f'median {mt["Quality"].median():.3f}   min {mt["Quality"].min():.3f}\n'
        f'{n_lo}/{N} below {QUALITY_THRESHOLD:.2f}', transform=ax.transAxes,
        va='top', ha='left', fontsize=FS, color=INK)
panel(ax, 'a', dx=-.115, dy=1.01)

ax2 = fig.add_subplot(gs[1])
cols = [c for c in MT_MATRIX_COLS if c in set(mt['Clade'])]
rng = np.random.default_rng(0)
for i, c in enumerate(cols):
    v = mt.loc[mt['Clade'] == c, 'nFound'].values
    a = mt.loc[mt['Clade'] == c, 'Ancestry'].mode().iat[0]
    ax2.scatter(i + (rng.random(len(v)) - .5) * .46, v, s=8, lw=.25,
                edgecolors='white', color=ANC_COLORS[a], marker=ANC_MARKER[a], alpha=.9,
                zorder=3)
    ax2.plot([i - .30, i + .30], [np.median(v)] * 2, color=INK, lw=.9, zorder=4)
ax2.set_xticks(range(len(cols)))
ax2.set_xticklabels(cols, rotation=90, fontsize=FS)
for j, t in enumerate(ax2.get_xticklabels()):
    _a = mt.loc[mt['Clade'] == cols[j], 'Ancestry'].mode().iat[0]
    t.set_color(BAND if _a == 'Unassigned' else ANC_COLORS[_a])   # grey text fails 4.5:1
ax2.set_xlim(-.7, len(cols) - .3)
ax2.set_ylabel('Defining polymorphisms detected')
ax2.set_xlabel('mtDNA haplogroup', labelpad=2)
ax2.grid(axis='y', zorder=0)
ax2.set_axisbelow(True)
despine(ax2)
ax2.legend([plt.Line2D([], [], lw=0, marker=ANC_MARKER[a], color=ANC_COLORS[a], markersize=4.5)
            for a in ('East', 'West')],
           [ANC_LABEL[a] for a in ('East', 'West')], frameon=False, ncol=2,
           loc='upper right', bbox_to_anchor=(1.005, 1.12), handlelength=1.0,
           handleheight=.75, columnspacing=1.1, handletextpad=.45)
panel(ax2, 'b', dx=-.075, dy=1.01)

save_fig(fig, f"Supplementary_Fig_{SUPP_NUM['fig_call_quality']}_chrM_call_quality", OUT_DIR, dpi=DPI)
plt.show()

## 7 - Supplementary Data

The full tribe by sub-haplogroup matrix and the lineage-diversity estimates are tabulated
rather than plotted: at terminal resolution most cells hold a single observation, and
maternal haplogroup diversity is saturated (h close to 1) in every group, so the ranking
between tribes carries no information. Diversity is given as Nei's unbiased estimator with
its analytic standard deviation (Nei 1987, eq. 8.12), together with the number of distinct
lineages per individual, which differs by an order of magnitude between the two markers.

In [ ]:
def nei_h_sd(labels):
    # Nei 1987, eq. 8.12.
    x = pd.Series(list(labels)).dropna()
    n = len(x)
    if n < 2:
        return np.nan, np.nan
    p = x.value_counts(normalize=True).values
    s2, s3 = np.sum(p ** 2), np.sum(p ** 3)
    h = n / (n - 1) * (1 - s2)
    var = 2 / (n * (n - 1)) * (2 * (n - 2) * (s3 - s2 ** 2) + s2 - s2 ** 2)
    return float(h), float(np.sqrt(var))


def diversity_rows(df, marker):
    out = []
    for name, sub in [('Whole cohort', df)] + [(t, df[df['Tribe'] == t])
                                               for t in sorted(set(df['Tribe']))
                                               if t != 'NA'
                                               and (df['Tribe'] == t).sum() >= MIN_TRIBE_N]:
        h, sd = nei_h_sd(sub['Terminal'])
        top = sub['Terminal'].value_counts()
        out.append({'Marker': marker, 'Group': name, 'n': len(sub),
                    'lineages': sub['Terminal'].nunique(),
                    'lineages_per_n': round(sub['Terminal'].nunique() / len(sub), 2),
                    'h': round(h, 4), 'h_SD': round(sd, 4),
                    'top_lineage': top.index[0], 'top_n': int(top.iloc[0])})
    return out


hg_summary = (mt.groupby(['Major', 'Terminal', 'Ancestry'], observed=True)
                .size().rename('n').reset_index()
                .assign(macro=lambda d: d['Major'].map(MT_MACRO_OF),
                        pct=lambda d: (d['n'] / N * 100).round(2))
                .sort_values(['n', 'Major'], ascending=[False, True])
                .reset_index(drop=True))

anc_rows = []
for name, sub in groups:
    n = len(sub)
    k = int((sub['Ancestry'] == 'East').sum())
    w = int((sub['Ancestry'] == 'West').sum())
    _, lo, hi = wilson(k, n)
    anc_rows.append({'Group': name, 'n': n, 'East': k, 'West': w,
                     'Unresolved': n - k - w,
                     'East_pct': round(k / n * 100, 1),
                     'East_lo95': round(lo * 100, 1),
                     'East_hi95': round(hi * 100, 1)})

div = pd.DataFrame(diversity_rows(mt, 'mtDNA'))

qc = mt[[c for c in ['SampleID', 'Haplogroup', 'Terminal', 'Major', 'Ancestry',
                     'Quality', 'nFound', 'nNotFound', 'nRemaining'] if c in mt.columns]]

XLSX_OUT = f"Supplementary_Data_{SUPP_NUM['data']}_chrM.xlsx"

sheets = [
    ('01_per_sample_calls',      mt[LONG_COLS], ('SampleID', 'Haplogroup', 'Terminal')),
    ('02_haplogroup_counts',
     hg_summary[['macro', 'Major', 'Terminal', 'Ancestry', 'n', 'pct']],
     ('macro', 'Major', 'Terminal')),
    ('03_tribe_by_clade',        pd.crosstab(mt['Tribe'], mt['Clade']).reset_index(), ('Tribe',)),
    ('04_tribe_by_terminal',     pd.crosstab(mt['Tribe'], mt['Terminal']).reset_index(), ('Tribe',)),
    ('05_ancestry_composition',  pd.DataFrame(anc_rows), ('Group',)),
    ('06_lineage_diversity',     div, ('Marker', 'Group', 'top_lineage')),
    ('07_call_quality',          qc, ('SampleID', 'Haplogroup', 'Terminal')),
]

wb = Workbook()
for i, (name, frame, keys) in enumerate(sheets):
    ws = wb.active if i == 0 else wb.create_sheet()
    ws.title = name
    write_sheet(ws, frame.reset_index(drop=True), key_cols=keys)
wb.save(os.path.join(OUT_DIR, XLSX_OUT))
print('saved:', XLSX_OUT, f'({len(sheets)} sheets)')
print(div.to_string(index=False))

In [ ]:
from docx import Document
from docx.shared import Pt, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT

SUPPL_TABLES = [
    (SUPP_NUM['tab_haplogroups'], 'Mitochondrial haplogroup composition of the cohort, by major clade and '
         'terminal PhyloTree label.',
     hg_summary[['macro', 'Major', 'Terminal', 'Ancestry', 'n', 'pct']]),
    (SUPP_NUM['tab_composition'], 'Mitochondrial phylogeographic composition of the whole cohort and of each '
         'maternal tribal stratum, with Wilson 95% confidence intervals.',
     pd.DataFrame(anc_rows)),
    (SUPP_NUM['tab_diversity'], 'Mitochondrial lineage diversity by tribal stratum, Nei unbiased gene diversity '
         'over terminal lineages with analytic standard deviation.',
     div),
]

def _fmt(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ''
    if isinstance(v, (bool, np.bool_)):
        return 'yes' if v else 'no'
    if isinstance(v, (int, np.integer)):
        return f'{int(v):,}'
    if isinstance(v, (float, np.floating)):
        return f'{v:,.4f}'.rstrip('0').rstrip('.')
    return str(v)

def add_table(doc, df, caption, tag):
    p = doc.add_paragraph()
    r = p.add_run(f'{tag}  {caption}')
    r.bold = True
    r.font.size = Pt(9)
    r.font.name = 'Arial'
    p.paragraph_format.space_after = Pt(4)
    t = doc.add_table(rows=1, cols=len(df.columns))
    t.style = 'Table Grid'
    t.alignment = WD_TABLE_ALIGNMENT.CENTER
    for j, col in enumerate(df.columns):
        c = t.rows[0].cells[j]
        c.text = ''
        run = c.paragraphs[0].add_run(str(col))
        run.bold = True; run.font.size = Pt(8); run.font.name = 'Arial'
        c.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER
    for _, row in df.iterrows():
        cells = t.add_row().cells
        for j, v in enumerate(row):
            cells[j].text = ''
            run = cells[j].paragraphs[0].add_run(_fmt(v))
            run.font.size = Pt(8); run.font.name = 'Arial'
            cells[j].paragraphs[0].alignment = (
                WD_ALIGN_PARAGRAPH.LEFT if j == 0 else WD_ALIGN_PARAGRAPH.CENTER)
    doc.add_paragraph().paragraph_format.space_after = Pt(10)

doc = Document()
for s in doc.sections:
    s.left_margin = s.right_margin = Inches(0.7)
h = doc.add_paragraph(); hr = h.add_run('Supplementary Tables 05')
hr.bold = True; hr.font.size = Pt(12); hr.font.name = 'Arial'

for num, cap, df in SUPPL_TABLES:
    if df is None or len(df) == 0:
        print(f'  Supplementary Table {num}: empty, skipped'); continue
    add_table(doc, df, cap, f'Supplementary Table {num}.')

DOCX = os.path.join(OUT_DIR, 'Supplementary_Tables_05.docx')
doc.save(DOCX)
print(f'saved {os.path.basename(DOCX)}')

## 8 - Outputs

| File | Content |
|---|---|
| `Fig6_chrM_composition.tiff` / `.png` | Fig. 6a, b - haplogroup spectrum; composition by maternal tribe |
| `Supplementary_Fig_8_chrM_call_quality.tiff` / `.png` | Supplementary Fig. 8 - haplogrep3 quality and defining polymorphisms |
| `Supplementary_Data_5_chrM.xlsx` | Supplementary Data 5 |
| `Supplementary_Tables_05.docx` | Supplementary Tables 24-26 |
| `kz_219_chrM_lineage_long.tsv` | long-format calls for the paired mtDNA and chrY comparison |